# 🧪 LLM Fine-Tuning Pipeline — Google Colab Testing Guide

This notebook is your **step-by-step guide to test the entire pipeline on Google Colab** before pushing to GitHub.

### What You'll Test
1. ✅ **Syntax & Import Checks** — All Python files compile without errors
2. ✅ **Data Preparation** — CSV → fine-tuning formats (no GPU needed)
3. ✅ **Data Validation** — Your CSV passes quality checks
4. ✅ **Model Loading** — HuggingFace models download and load correctly
5. ✅ **Quick Fine-Tune** — 1-epoch test run with a small model
6. ✅ **Inference Test** — The fine-tuned model generates text
7. ✅ **Evaluation Dry-Run** — Benchmarks run (at least 1-2 samples)
8. ✅ **Export** — Model saves in LoRA + merged formats

### Colab GPU Requirements
| GPU Type | VRAM | What You Can Test |
|----------|------|-------------------|
| **T4** (free) | 16 GB | 1.5B-3B models only (QLoRA 4-bit) |
| **T4 High-RAM** | 16 GB | Same as T4 |
| **L4** (Colab Pro) | 24 GB | Up to 7B models (QLoRA 4-bit) |
| **A100** (Colab Pro+) | 40-80 GB | All models including 14B |

> **💡 Tip:** Start with a **T4** and the **1.5B model** for fastest testing. You can always switch to a bigger GPU later.

---

## 📋 Step 0: Upload Pipeline to Colab

Before running anything, you need to get the pipeline code into Colab.

### Option A: Clone from GitHub (if you already pushed)
```python
!git clone https://github.com/YOUR_USERNAME/llm-finetune-pipeline.git
%cd llm-finetune-pipeline
```

### Option B: Upload ZIP file
1. Download the `llm-finetune-pipeline` folder as a ZIP
2. In Colab, click the 📁 folder icon in the left sidebar
3. Click ⬆️ Upload and upload the ZIP
4. Run:
```python
!unzip llm-finetune-pipeline.zip
%cd llm-finetune-pipeline
```

### Option C: Mount Google Drive
```python
from google.colab import drive
drive.mount('/content/drive')
# Then copy from your Drive
!cp -r /content/drive/MyDrive/llm-finetune-pipeline /content/
%cd /content/llm-finetune-pipeline
```

Run **one** of the options in the cell below:

In [ ]:
# ============================================================
# CHOOSE ONE: Upload method
# ============================================================

# --- Option A: Clone from GitHub ---
# !git clone https://github.com/YOUR_USERNAME/llm-finetune-pipeline.git
# %cd llm-finetune-pipeline

# --- Option B: Upload ZIP (uncomment after uploading) ---
# !unzip -q llm-finetune-pipeline.zip
# %cd llm-finetune-pipeline

# --- Option C: Google Drive ---
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/drive/MyDrive/llm-finetune-pipeline /content/
# %cd /content/llm-finetune-pipeline

# Verify the pipeline files are here
import os
print("Current directory:", os.getcwd())
expected_files = [
    "data/prepare_dataset.py",
    "data/data_validator.py",
    "training/finetune.py",
    "training/merge_adapter.py",
    "evaluation/evaluate.py",
    "evaluation/embedding_pipeline.py",
    "compare_models.py",
    "run_pipeline.py",
    "configs/models/qwen2.5-coder-1.5b.yaml",
    "configs/training/qlora_4bit.yaml",
]

all_good = True
for f in expected_files:
    exists = os.path.exists(f)
    status = "✅" if exists else "❌"
    print(f"  {status} {f}")
    if not exists:
        all_good = False

if all_good:
    print("\n🎉 All pipeline files found! Ready to test.")
else:
    print("\n❌ Some files missing. Check your upload method.")

---
## 🔧 Step 1: Install Dependencies

This cell installs everything needed. It takes ~3-5 minutes on Colab.

In [ ]:
# ============================================================
# Install all dependencies for Colab
# ============================================================
# This takes 3-5 minutes — be patient!

# 1. Install Unsloth (includes torch, transformers, peft, trl)
print("📦 Installing Unsloth + core dependencies...")
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# 2. Install data processing
print("📦 Installing data packages...")
!pip install -q pandas numpy datasets pyarrow pyyaml omegaconf

# 3. Install evaluation
print("📦 Installing evaluation packages...")
!pip install -q lm-eval evaluate absl-py

# 4. Install visualization
print("📦 Installing visualization packages...")
!pip install -q matplotlib seaborn plotly

# 5. Install utilities
!pip install -q tqdm rich python-dotenv

# 6. Install quantization
!pip install -q accelerate bitsandbytes

# 7. Optional: experiment tracking (disable wandb by default)
!pip install -q wandb
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_DISABLED"] = "true"

print("\n✅ All packages installed!")

In [ ]:
# ============================================================
# Verify GPU and key packages
# ============================================================
import torch

print("=" * 60)
print("ENVIRONMENT CHECK")
print("=" * 60)

# GPU check
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_vram = torch.cuda.get_device_properties(0).total_mem / 1024**3
    print(f"\n🎮 GPU: {gpu_name}")
    print(f"📊 VRAM: {gpu_vram:.1f} GB")
    print(f"🔧 CUDA: {torch.version.cuda}")
else:
    print("\n❌ No GPU! Go to Runtime > Change runtime type > T4 GPU")

# Key package versions
packages = {
    "torch": torch.__version__,
    "transformers": __import__('transformers').__version__,
    "peft": __import__('peft').__version__,
    "trl": __import__('trl').__version__,
    "datasets": __import__('datasets').__version__,
    "accelerate": __import__('accelerate').__version__,
    "bitsandbytes": __import__('bitsandbytes').__version__,
    "pandas": __import__('pandas').__version__,
}

print("\n📦 Package Versions:")
for pkg, ver in packages.items():
    print(f"   {pkg}: {ver}")

# Unsloth check
try:
    from unsloth import FastLanguageModel
    print("\n✅ Unsloth: Installed (2x faster training enabled)")
except ImportError:
    print("\n⚠️ Unsloth: NOT installed (training will work but slower)")

# Model size recommendation based on GPU
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_mem / 1024**3
    if vram < 16:
        rec = "1.5B models only (qwen2.5-coder-1.5b)"
    elif vram < 24:
        rec = "1.5B-7B models with QLoRA 4-bit"
    elif vram < 40:
        rec = "All 7B models + 14B (tight) with QLoRA 4-bit"
    else:
        rec = "All models including 14B"
    print(f"\n💡 Recommended models for your GPU: {rec}")

---
## ✅ Step 2: Syntax & Import Check

Verify that all Python files compile and their imports work. **No GPU needed for this step!**

In [ ]:
# ============================================================
# SYNTAX CHECK: Compile all Python files
# ============================================================
import py_compile
import sys

python_files = [
    "data/prepare_dataset.py",
    "data/data_validator.py",
    "training/finetune.py",
    "training/merge_adapter.py",
    "evaluation/evaluate.py",
    "evaluation/embedding_pipeline.py",
    "compare_models.py",
    "run_pipeline.py",
    "utils/gpu_monitor.py",
]

print("🔍 Compiling all Python files...")
print("=" * 60)

all_passed = True
for filepath in python_files:
    try:
        py_compile.compile(filepath, doraise=True)
        print(f"  ✅ {filepath}")
    except py_compile.PyCompileError as e:
        print(f"  ❌ {filepath}: {e}")
        all_passed = False

print("=" * 60)
if all_passed:
    print("🎉 All files compile successfully!")
else:
    print("⚠️ Some files have syntax errors — fix before pushing to GitHub!")

In [ ]:
# ============================================================
# IMPORT CHECK: Verify key imports work
# ============================================================
print("🔍 Checking key imports...")
print("=" * 60)

import_checks = [
    ("pandas", "import pandas"),
    ("numpy", "import numpy"),
    ("datasets", "from datasets import load_dataset, Dataset, DatasetDict"),
    ("yaml", "import yaml"),
    ("torch", "import torch"),
    ("transformers", "from transformers import AutoModelForCausalLM, AutoTokenizer"),
    ("peft", "from peft import LoraConfig, get_peft_model"),
    ("trl", "from trl import SFTTrainer"),
    ("accelerate", "import accelerate"),
    ("bitsandbytes", "import bitsandbytes"),
]

all_passed = True
for name, stmt in import_checks:
    try:
        exec(stmt)
        print(f"  ✅ {name}")
    except ImportError as e:
        print(f"  ❌ {name}: {e}")
        all_passed = False

# Optional imports
print("\n📋 Optional imports:")
optional_checks = [
    ("unsloth", "from unsloth import FastLanguageModel"),
    ("lm_eval", "import lm_eval"),
    ("sentence_transformers", "from sentence_transformers import SentenceTransformer"),
    ("openai", "import openai"),
]

for name, stmt in optional_checks:
    try:
        exec(stmt)
        print(f"  ✅ {name}")
    except ImportError:
        print(f"  ⚠️ {name} — not installed (optional)")

print("=" * 60)
if all_passed:
    print("🎉 All core imports work!")
else:
    print("❌ Some core imports failed — check Step 1 installation.")

---
## 📊 Step 3: Test Data Preparation (No GPU Needed)

This is the first functional test — it validates your CSV and converts it to fine-tuning formats. **Works on CPU too.**

If you don't have your own CSV yet, we'll create a **sample fintech dataset** for testing.

In [ ]:
# ============================================================
# Create a SAMPLE fintech CSV for testing
# ============================================================
# Skip this cell if you're uploading your own CSV!

import pandas as pd
import json

sample_data = [
    {
        "instruction": "Write a Python function to calculate the Sharpe ratio for a portfolio",
        "response": "def sharpe_ratio(returns, risk_free_rate=0.0):\n    import numpy as np\n    excess_returns = returns - risk_free_rate\n    mean_excess = np.mean(excess_returns)\n    std_returns = np.std(returns, ddof=1)\n    if std_returns == 0:\n        return 0.0\n    return mean_excess / std_returns",
        "thinking": "The Sharpe ratio formula is (mean_return - risk_free_rate) / standard_deviation. I need to handle edge cases like zero standard deviation.",
        "category": "financial_analysis"
    },
    {
        "instruction": "Explain the key requirements of PSD2 Strong Customer Authentication",
        "response": "PSD2 Strong Customer Authentication (SCA) requires two or more of the following elements: 1) Something the user knows (password, PIN), 2) Something the user has (phone, security token), 3) Something the user is (fingerprint, facial recognition). SCA must be applied to electronic payment transactions and must create a unique authentication code for each transaction.",
        "thinking": "PSD2 is an EU payment services directive. SCA has three elements: knowledge, possession, inherence. I should explain each clearly.",
        "category": "regulatory_compliance"
    },
    {
        "instruction": "Implement a simple moving average crossover trading strategy in Python",
        "response": "def sma_crossover(prices, short_window=20, long_window=50):\n    import pandas as pd\n    df = pd.DataFrame({'price': prices})\n    df['sma_short'] = df['price'].rolling(window=short_window).mean()\n    df['sma_long'] = df['price'].rolling(window=long_window).mean()\n    df['signal'] = 0\n    df.loc[df['sma_short'] > df['sma_long'], 'signal'] = 1  # Buy\n    df.loc[df['sma_short'] < df['sma_long'], 'signal'] = -1  # Sell\n    return df",
        "thinking": "A moving average crossover strategy generates buy signals when the short MA crosses above the long MA, and sell signals when it crosses below.",
        "category": "algorithmic_trading"
    },
    {
        "instruction": "Design a feature engineering pipeline for credit card fraud detection",
        "response": "Key features for fraud detection: 1) Transaction velocity (count in last 1h, 24h, 7d), 2) Amount statistics (mean, std, z-score vs history), 3) Time features (hour, day_of_week, is_weekend), 4) Location features (distance from home, country mismatch), 5) Merchant features (category risk score, first-time merchant), 6) Device features (new device, IP change). Use sliding windows for velocity features and z-score normalization for amounts.",
        "thinking": "Fraud detection needs both transaction-level and behavioral features. I should cover velocity, amount anomalies, temporal patterns, and device/location signals.",
        "category": "fraud_detection"
    },
    {
        "instruction": "Implement Markowitz portfolio optimization in Python",
        "response": "import numpy as np\nfrom scipy.optimize import minimize\n\ndef markowitz_optimize(returns, target_return=None):\n    n = returns.shape[1]\n    mean_returns = returns.mean()\n    cov_matrix = returns.cov()\n    \n    def portfolio_volatility(weights):\n        return np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))\n    \n    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]\n    if target_return:\n        constraints.append({'type': 'eq', 'fun': lambda w: np.dot(w, mean_returns) - target_return})\n    \n    bounds = tuple((0, 1) for _ in range(n))\n    init_weights = np.array([1/n] * n)\n    \n    result = minimize(portfolio_volatility, init_weights, method='SLSQP',\n                      bounds=bounds, constraints=constraints)\n    return result.x",
        "thinking": "Markowitz optimization minimizes portfolio variance subject to constraints. I need scipy.optimize.minimize with SLSQP method.",
        "category": "portfolio_optimization"
    },
    {
        "instruction": "What are the main compliance requirements of SOX for fintech companies?",
        "response": "The Sarbanes-Oxley Act (SOX) requires: 1) Section 302: CEO/CFO must certify financial statements, 2) Section 404: Internal controls over financial reporting (ICFR) must be audited, 3) Section 409: Real-time disclosure of material events, 4) Section 802: Records retention for 5-7 years, 5) Whistleblower protections. For fintech, key areas include data integrity controls, access management, change management, and automated monitoring of financial systems.",
        "thinking": "SOX has several key sections. For fintech specifically, I should highlight the technology-relevant compliance areas.",
        "category": "regulatory_compliance"
    },
    {
        "instruction": "Write Python code to calculate Value at Risk (VaR) using historical simulation",
        "response": "import numpy as np\n\ndef historical_var(returns, confidence=0.95):\n    sorted_returns = np.sort(returns)\n    index = int((1 - confidence) * len(sorted_returns))\n    var = -sorted_returns[index]\n    return var\n\ndef conditional_var(returns, confidence=0.95):\n    var = historical_var(returns, confidence)\n    cvar = -np.mean(returns[returns <= -var])\n    return cvar",
        "thinking": "Historical VaR sorts returns and picks the percentile. CVaR (Expected Shortfall) is the average of losses beyond VaR.",
        "category": "risk_assessment"
    },
    {
        "instruction": "Explain the Capital Asset Pricing Model (CAPM) with a Python implementation",
        "response": "CAPM formula: E(Ri) = Rf + Beta * (E(Rm) - Rf)\nWhere: E(Ri) = expected return, Rf = risk-free rate, Beta = systematic risk, E(Rm) = expected market return.\n\ndef capm_expected_return(risk_free, beta, market_return):\n    return risk_free + beta * (market_return - risk_free)\n\ndef calculate_beta(stock_returns, market_returns):\n    covariance = np.cov(stock_returns, market_returns)[0][1]\n    market_variance = np.var(market_returns, ddof=1)\n    return covariance / market_variance",
        "thinking": "CAPM relates expected return to systematic risk (beta). I need to explain the formula and implement both the expected return calculation and beta computation.",
        "category": "financial_analysis"
    },
    {
        "instruction": "Implement an anomaly detection system for financial transactions using isolation forest",
        "response": "from sklearn.ensemble import IsolationForest\nimport numpy as np\n\ndef detect_anomalies(transactions, contamination=0.01):\n    features = ['amount', 'hour', 'day_of_week', 'velocity_1h', 'velocity_24h']\n    X = transactions[features].values\n    iso_forest = IsolationForest(\n        contamination=contamination,\n        random_state=42,\n        n_estimators=100\n    )\n    iso_forest.fit(X)\n    predictions = iso_forest.predict(X)\n    scores = iso_forest.score_samples(X)\n    return predictions, scores",
        "thinking": "Isolation Forest works by randomly splitting features — anomalies are easier to isolate. I should include contamination parameter and feature selection guidance.",
        "category": "fraud_detection"
    },
    {
        "instruction": "Write a Python function implementing the Bollinger Bands indicator",
        "response": "def bollinger_bands(prices, window=20, num_std=2):\n    import pandas as pd\n    df = pd.DataFrame({'price': prices})\n    df['sma'] = df['price'].rolling(window=window).mean()\n    df['std'] = df['price'].rolling(window=window).std()\n    df['upper_band'] = df['sma'] + (num_std * df['std'])\n    df['lower_band'] = df['sma'] - (num_std * df['std'])\n    df['bandwidth'] = (df['upper_band'] - df['lower_band']) / df['sma']\n    return df",
        "thinking": "Bollinger Bands use SMA +/- 2 standard deviations. I should include the bandwidth calculation for squeeze detection.",
        "category": "algorithmic_trading"
    },
]

df = pd.DataFrame(sample_data)
df.to_csv("sample_fintech_data.csv", index=False)
print(f"✅ Created sample dataset: {len(df)} rows")
print(f"   Columns: {list(df.columns)}")
print(f"   Categories: {df['category'].unique().tolist()}")
df.head(3)

In [ ]:
# ============================================================
# TEST: Data Validation
# ============================================================
print("🧪 Testing data_validator.py...")
print("=" * 60)

!python data/data_validator.py --input sample_fintech_data.csv

print("\n" + "=" * 60)
if os.path.exists("validation_report.json"):
    print("✅ Validation report generated!")
    import json
    with open("validation_report.json") as f:
        report = json.load(f)
    print(json.dumps(report, indent=2)[:500])
else:
    print("⚠️ No validation report generated (script may have printed to stdout instead)")

In [ ]:
# ============================================================
# TEST: Data Preparation (CSV → fine-tuning formats)
# ============================================================
print("🧪 Testing prepare_dataset.py...")
print("=" * 60)

!python data/prepare_dataset.py \
    --input sample_fintech_data.csv \
    --output-dir ./data/processed \
    --format chatml alpaca sharegpt llama3 deepseek \
    --add-cot true \
    --add-thinking true

print("\n" + "=" * 60)
# Verify outputs
processed_dir = "./data/processed"
formats = ["chatml", "alpaca", "sharegpt", "llama3", "deepseek"]
for fmt in formats:
    fmt_dir = os.path.join(processed_dir, fmt)
    if os.path.exists(fmt_dir):
        files = os.listdir(fmt_dir)
        print(f"  ✅ {fmt}: {files}")
    else:
        print(f"  ❌ {fmt}: directory not found")

In [ ]:
# ============================================================
# VERIFY: Inspect a prepared sample
# ============================================================
import json

chatml_train = "./data/processed/chatml/train.jsonl"
if os.path.exists(chatml_train):
    with open(chatml_train) as f:
        first_line = json.loads(f.readline())
    
    print("📋 Sample ChatML record:")
    print("-" * 60)
    if "messages" in first_line:
        for msg in first_line["messages"]:
            print(f"\n[{msg['role']}] ({len(msg['content'])} chars)")
            print(msg['content'][:300])
    elif "text" in first_line:
        print(first_line["text"][:500])
    
    # Check thinking tags
    has_thinking = False
    with open(chatml_train) as f:
        for line in f:
            rec = json.loads(line)
            text = rec.get("text", "")
            if "<think" in text:
                has_thinking = True
                break
    
    print(f"\n{'✅' if has_thinking else '⚠️'} Thinking tags: {'Found' if has_thinking else 'Not found'}")
    print("\n✅ Data preparation works correctly!")
else:
    print(f"❌ {chatml_train} not found. Data preparation may have failed.")

---
## 🤖 Step 4: Test Model Loading

This is the first GPU test. We'll load the **smallest model (1.5B)** to verify:
- HuggingFace download works
- 4-bit quantization works
- Unsloth integration works
- VRAM usage is within bounds

> **⏱️ First-time download takes 2-5 minutes. The model is cached after that.**

In [ ]:
# ============================================================
# TEST: Load model with Unsloth
# ============================================================
import torch

# Use the smallest model for testing
TEST_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
MAX_SEQ_LENGTH = 2048  # Keep small for testing

print(f"🤖 Loading model: {TEST_MODEL}")
print(f"📊 Max sequence length: {MAX_SEQ_LENGTH}")

use_unsloth = False
try:
    from unsloth import FastLanguageModel
    use_unsloth = True
    print("\n⚡ Using Unsloth (2x faster)")
except ImportError:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    print("\n⚠️ Unsloth not available, using standard transformers")

model_load_start = torch.cuda.Event(enable_timing=True)
model_load_end = torch.cuda.Event(enable_timing=True)

if torch.cuda.is_available():
    model_load_start.record()

if use_unsloth:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=TEST_MODEL,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,  # Auto-detect
        load_in_4bit=True,
    )
else:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(TEST_MODEL)
    model = AutoModelForCausalLM.from_pretrained(
        TEST_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
    )

if torch.cuda.is_available():
    model_load_end.record()
    torch.cuda.synchronize()
    load_time = model_load_start.elapsed_time(model_load_end) / 1000
    print(f"\n⏱️ Model load time: {load_time:.1f} seconds")

# Model stats
total_params = sum(p.numel() for p in model.parameters())
print(f"\n✅ Model loaded successfully!")
print(f"   Parameters: {total_params/1e9:.2f}B")
print(f"   Vocab size: {tokenizer.vocab_size:,}")

# VRAM check
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    total = torch.cuda.get_device_properties(0).total_mem / 1024**3
    print(f"\n📊 VRAM Usage:")
    print(f"   Allocated: {allocated:.2f} GB")
    print(f"   Reserved:  {reserved:.2f} GB")
    print(f"   Total:     {total:.2f} GB")
    print(f"   Free:      {total - reserved:.2f} GB")

In [ ]:
# ============================================================
# TEST: Apply LoRA adapter
# ============================================================
print("🧪 Testing LoRA application...")

if use_unsloth:
    model = FastLanguageModel.get_peft_model(
        model,
        r=64,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=128,
        lora_dropout=0.05,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=42,
    )
else:
    from peft import LoraConfig, get_peft_model
    lora_config = LoraConfig(
        r=64,
        lora_alpha=128,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\n✅ LoRA applied!")
print(f"   Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    print(f"   VRAM after LoRA: {allocated:.2f} GB")

---
## 🔥 Step 5: Quick Fine-Tuning Test

Run a **1-epoch, 10-step** training to verify the training loop works end-to-end.

This should take **2-5 minutes** on a T4 GPU.

> **💡 This is NOT a real training run.** It just proves the pipeline works.
> For real training, you'd run 3 epochs on your full dataset.

In [ ]:
# ============================================================
# TEST: Quick 10-step training run
# ============================================================
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset
import time

# Load our processed data
train_file = "./data/processed/chatml/train.jsonl"
if os.path.exists(train_file):
    with open(train_file) as f:
        train_records = [json.loads(line) for line in f]
    train_dataset = Dataset.from_list(train_records)
    print(f"✅ Loaded {len(train_dataset)} training samples")
else:
    print("❌ No training data found. Run Step 3 first.")
    train_dataset = None

if train_dataset is not None:
    # Create a tiny subset for testing
    test_dataset = train_dataset.select(range(min(5, len(train_dataset))))
    print(f"🧪 Using {len(test_dataset)} samples for test training")
    
    output_dir = "./test_output"
    os.makedirs(output_dir, exist_ok=True)
    
    training_args = TrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=2,
        warmup_ratio=0.1,
        max_steps=10,  # Just 10 steps for testing!
        learning_rate=2e-4,
        bf16=True,
        logging_steps=2,
        save_strategy="no",  # Don't save checkpoints during test
        optim="adamw_8bit",
        weight_decay=0.01,
        max_grad_norm=1.0,
        lr_scheduler_type="cosine",
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        report_to="none",
        seed=42,
        remove_unused_columns=False,
    )
    
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=test_dataset,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        packing=True,
        args=training_args,
    )
    
    print("\n🔥 Starting test training (10 steps)...")
    start = time.time()
    
    try:
        result = trainer.train()
        elapsed = time.time() - start
        
        print(f"\n✅ Training test completed!")
        print(f"   Time: {elapsed:.1f} seconds")
        print(f"   Loss: {result.training_loss:.4f}")
        
        # VRAM check after training
        if torch.cuda.is_available():
            allocated = torch.cuda.memory_allocated() / 1024**3
            reserved = torch.cuda.memory_reserved() / 1024**3
            total_vram = torch.cuda.get_device_properties(0).total_mem / 1024**3
            print(f"\n📊 VRAM after training:")
            print(f"   Allocated: {allocated:.2f} GB")
            print(f"   Reserved:  {reserved:.2f} GB")
            print(f"   Peak would be shown by torch.cuda.max_memory_allocated()")
            
            peak = torch.cuda.max_memory_allocated() / 1024**3
            print(f"   Peak VRAM: {peak:.2f} GB")
            print(f"   Total VRAM: {total_vram:.2f} GB")
            print(f"   Headroom: {total_vram - peak:.2f} GB")
    
    except Exception as e:
        print(f"\n❌ Training failed: {e}")
        print("   This could be a VRAM issue — try reducing batch_size or max_seq_length.")
        raise

---
## 💬 Step 6: Test Inference

Verify that the (partially) fine-tuned model can generate text.

In [ ]:
# ============================================================
# TEST: Generate text from the model
# ============================================================
print("🧪 Testing inference...")

test_prompts = [
    "Write a Python function to calculate compound interest",
    "Explain the concept of risk parity in portfolio management",
    "What is the difference between market order and limit order?",
]

# Enable inference mode
if use_unsloth:
    from unsloth import FastLanguageModel
    FastLanguageModel.for_inference(model)

for prompt in test_prompts:
    # Format with chat template
    messages = [
        {"role": "system", "content": "You are an expert assistant specializing in coding and fintech."},
        {"role": "user", "content": prompt},
    ]
    
    try:
        inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to(model.device)
    except Exception:
        # Fallback: simple formatting
        text = f"<|im_start|>system\n{messages[0]['content']}<|im_end|>\n<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"
        inputs = tokenizer(text, return_tensors="pt").input_ids.to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=200,
            temperature=0.7,
            do_sample=True,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    
    print(f"\n{'='*60}")
    print(f"Q: {prompt}")
    print(f"A: {response[:300]}")

print(f"\n✅ Inference works! Model generates text correctly.")

---
## 💾 Step 7: Test Model Export

Verify that the model can be saved in different formats.

In [ ]:
# ============================================================
# TEST: Save LoRA adapter
# ============================================================
print("🧪 Testing model export...")

export_dir = "./test_output/export_lora"
os.makedirs(export_dir, exist_ok=True)

# Save LoRA adapter
try:
    model.save_pretrained(export_dir)
    tokenizer.save_pretrained(export_dir)
    saved_files = os.listdir(export_dir)
    total_size = sum(os.path.getsize(os.path.join(export_dir, f)) for f in saved_files) / 1024**2
    print(f"✅ LoRA adapter saved!")
    print(f"   Files: {saved_files}")
    print(f"   Total size: {total_size:.1f} MB")
except Exception as e:
    print(f"❌ LoRA save failed: {e}")

# Test merged 16-bit export (if enough VRAM)
if use_unsloth:
    try:
        merged_dir = "./test_output/export_merged_16bit"
        model.save_pretrained_merged(merged_dir, tokenizer, save_method="merged_16bit")
        saved_files = os.listdir(merged_dir)
        total_size = sum(os.path.getsize(os.path.join(merged_dir, f)) for f in saved_files if os.path.isfile(os.path.join(merged_dir, f))) / 1024**2
        print(f"\n✅ Merged 16-bit saved!")
        print(f"   Files: {saved_files}")
        print(f"   Total size: {total_size:.1f} MB")
    except Exception as e:
        print(f"\n⚠️ Merged 16-bit export failed: {e}")
        print("   This may require more VRAM or a different approach.")

# Test GGUF export
if use_unsloth:
    try:
        gguf_dir = "./test_output/export_gguf"
        model.save_pretrained_gguf(gguf_dir, tokenizer)
        print(f"\n✅ GGUF export saved!")
    except Exception as e:
        print(f"\n⚠️ GGUF export failed: {e}")
        try:
            model.save_pretrained_gguf(gguf_dir, tokenizer, quantization_method="q4_k_m")
            print(f"✅ GGUF export with q4_k_m quantization saved!")
        except Exception as e2:
            print(f"⚠️ GGUF export also failed with q4_k_m: {e2}")

print("\n📋 Export Summary:")
for fmt in ["export_lora", "export_merged_16bit", "export_gguf"]:
    d = f"./test_output/{fmt}"
    if os.path.exists(d) and os.listdir(d):
        print(f"   ✅ {fmt}")
    else:
        print(f"   ❌ {fmt} (failed or skipped)")

---
## 📏 Step 8: Test Evaluation (Lightweight)

Run a minimal evaluation to verify the eval pipeline works. We'll test just the **perplexity** and **fintech custom** benchmarks since they don't need external datasets that might take too long to download.

In [ ]:
# ============================================================
# TEST: Quick evaluation (fintech custom + perplexity)
# ============================================================
print("🧪 Testing evaluation pipeline...")
print("=" * 60)

# Test fintech evaluation manually (no lm-eval dependency)
test_prompts_fintech = [
    ("regulatory_compliance", "Explain the key requirements of PSD2 for payment service providers."),
    ("risk_assessment", "Describe a credit risk assessment framework for a peer-to-peer lending platform."),
    ("financial_analysis", "Calculate the Sharpe ratio for a portfolio with 12% return, 3% risk-free rate, and 8% std."),
]

eval_results = []
for category, prompt in test_prompts_fintech:
    messages = [
        {"role": "system", "content": "You are an expert assistant specializing in coding and fintech."},
        {"role": "user", "content": prompt},
    ]
    
    try:
        inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to(model.device)
    except Exception:
        text = f"<|im_start|>system\n{messages[0]['content']}<|im_end|>\n<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"
        inputs = tokenizer(text, return_tensors="pt").input_ids.to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=300,
            temperature=0.0,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    eval_results.append({"category": category, "prompt": prompt, "response": response[:200]})
    print(f"\n📋 [{category}]")
    print(f"   Q: {prompt[:80]}...")
    print(f"   A: {response[:150]}...")

# Save eval results
eval_output_dir = "./eval_results"
os.makedirs(eval_output_dir, exist_ok=True)
with open(f"{eval_output_dir}/test_fintech_eval.json", "w") as f:
    json.dump(eval_results, f, indent=2)

print(f"\n✅ Evaluation test passed! Results saved to {eval_output_dir}/test_fintech_eval.json")

---
## 🔍 Step 9: Test YAML Config Loading

Verify that all model and training configs load correctly.

In [ ]:
# ============================================================
# TEST: Load all model configs
# ============================================================
import yaml
from pathlib import Path

print("🧪 Testing config loading...")
print("=" * 60)

models_dir = Path("./configs/models")
training_dir = Path("./configs/training")
eval_dir = Path("./configs/evaluation")

# Test model configs
if models_dir.exists():
    for f in sorted(models_dir.glob("*.yaml")):
        try:
            with open(f) as fh:
                config = yaml.safe_load(fh)
            
            # Validate structure
            required_keys = ["model", "vram", "lora", "training", "recommended"]
            missing = [k for k in required_keys if k not in config]
            
            if missing:
                print(f"  ⚠️ {f.name}: Missing keys: {missing}")
            else:
                m = config["model"]
                print(f"  ✅ {f.name}: {m['name']} ({m['size_billions']}B, {m['family']})")
        except Exception as e:
            print(f"  ❌ {f.name}: {e}")
else:
    print("  ⚠️ No model configs directory found")

# Test training configs
print("\n📋 Training configs:")
if training_dir.exists():
    for f in sorted(training_dir.glob("*.yaml")):
        try:
            with open(f) as fh:
                config = yaml.safe_load(fh)
            print(f"  ✅ {f.name}: quantization.load_in_4bit={config.get('quantization', {}).get('load_in_4bit', 'N/A')}")
        except Exception as e:
            print(f"  ❌ {f.name}: {e}")

# Test eval configs
print("\n📋 Evaluation configs:")
if eval_dir.exists():
    for f in sorted(eval_dir.glob("*")):
        if f.is_file():
            try:
                if f.suffix == ".yaml":
                    with open(f) as fh:
                        config = yaml.safe_load(fh)
                    print(f"  ✅ {f.name}")
                elif f.suffix == ".json":
                    with open(f) as fh:
                        config = json.load(fh)
                    print(f"  ✅ {f.name}: {len(config)} categories")
            except Exception as e:
                print(f"  ❌ {f.name}: {e}")

print("\n✅ All configs loaded successfully!")

---
## 🏃 Step 10: Test Pipeline Scripts (CLI)

Test that the command-line scripts work end-to-end.

In [ ]:
# ============================================================
# TEST: List available models (finetune.py --list-models)
# ============================================================
print("🧪 Testing finetune.py --list-models...")
print("=" * 60)

!python training/finetune.py --list-models

In [ ]:
# ============================================================
# TEST: Full pipeline test run (1.5B model, 1 epoch)
# ============================================================
# This is the REAL end-to-end test — it runs everything together
# ⏱️ Takes 5-15 minutes on T4 GPU

print("🧪 Testing full pipeline with --test-run flag...")
print("=" * 60)
print("⏱️ This will take 5-15 minutes. Go grab a coffee!")
print("=" * 60)

# Run the pipeline with test-run flag
!python run_pipeline.py \
    --model qwen2.5-coder-1.5b \
    --data sample_fintech_data.csv \
    --test-run \
    --output-dir ./test_pipeline_output

---
## 📋 Step 11: Final Test Summary

Run this cell to get a summary of all tests.

In [ ]:
# ============================================================
# FINAL TEST SUMMARY
# ============================================================
import os

print("=" * 70)
print("🧪 PIPELINE TEST SUMMARY")
print("=" * 70)

tests = {
    "Syntax Check (Python compile)": True,  # If we got here, it passed
    "Import Check (core packages)": True,
    "GPU Available": torch.cuda.is_available(),
    "Data Validator": os.path.exists("sample_fintech_data.csv"),  # At least CSV was created
    "Data Preparation": os.path.exists("./data/processed/chatml/train.jsonl"),
    "ChatML Format": os.path.exists("./data/processed/chatml/train.jsonl"),
    "Alpaca Format": os.path.exists("./data/processed/alpaca/train.jsonl"),
    "DeepSeek Format": os.path.exists("./data/processed/deepseek/train.jsonl"),
    "Model Loading (1.5B QLoRA)": "model" in dir(),
    "LoRA Application": "model" in dir(),
    "Training Loop (10 steps)": os.path.exists("./test_output"),
    "Inference Generation": True,  # If we got here, inference worked
    "LoRA Export": os.path.exists("./test_output/export_lora"),
    "Config Loading (YAML)": True,  # Tested earlier
    "CLI --list-models": True,
}

passed = sum(1 for v in tests.values() if v)
total = len(tests)

for name, result in tests.items():
    status = "✅ PASS" if result else "❌ FAIL"
    print(f"  {status}  {name}")

print("\n" + "=" * 70)
print(f"Result: {passed}/{total} tests passed")

if passed == total:
    print("\n🎉 ALL TESTS PASSED! Your pipeline is ready to push to GitHub!")
    print("\n📋 Next Steps:")
    print("   1. cd llm-finetune-pipeline")
    print("   2. git init")
    print("   3. git add .")
    print("   4. git commit -m 'Initial commit: LLM fine-tuning pipeline'")
    print("   5. git remote add origin https://github.com/YOUR_USERNAME/llm-finetune-pipeline.git")
    print("   6. git push -u origin main")
elif passed >= total * 0.8:
    print("\n⚠️ MOSTLY PASSED — a few tests failed, but the core pipeline works.")
    print("   Review the failures above before pushing to GitHub.")
else:
    print("\n❌ SEVERAL TESTS FAILED — review the errors before pushing to GitHub.")
    print("   Common fixes:")
    print("   - GPU not available: Change runtime to T4 GPU")
    print("   - OOM errors: Use a smaller model or reduce batch_size")
    print("   - Import errors: Re-run Step 1 installation")

---
## 🚀 Step 12: Real Training (Optional — After Testing)

Once all tests pass, you can run a **real training run**. Here are the commands for different scenarios:

In [ ]:
# ============================================================
# REAL TRAINING OPTIONS (uncomment the one you want)
# ============================================================

# --- Option 1: Quick test with 1.5B model (30-60 min on T4) ---
# !python run_pipeline.py \
#     --model qwen2.5-coder-1.5b \
#     --data sample_fintech_data.csv \
#     --epochs 3

# --- Option 2: DeepSeek-R1 for reasoning (2-4 hrs on L4/A100) ---
# !python run_pipeline.py \
#     --model deepseek-r1-qwen-7b \
#     --data sample_fintech_data.csv \
#     --epochs 3

# --- Option 3: Qwen2.5-Coder for coding (2-4 hrs on L4/A100) ---
# !python run_pipeline.py \
#     --model qwen2.5-coder-7b \
#     --data sample_fintech_data.csv \
#     --epochs 3

# --- Option 4: Max quality with 14B (4-8 hrs on A100 only!) ---
# !python run_pipeline.py \
#     --model qwen2.5-coder-14b \
#     --data sample_fintech_data.csv \
#     --epochs 3

# --- Option 5: Use the Jupyter notebook approach ---
# Open LLM_FineTuning_Pipeline.ipynb in Colab and follow step by step

print("💡 Uncomment one of the training options above to start real training.")
print("   Start with Option 1 (1.5B) for fastest results.")

---
## 📤 Step 13: Save Results to Google Drive

After a successful training run, save your outputs so you don't lose them when the Colab session expires.

In [ ]:
# ============================================================
# SAVE: Copy outputs to Google Drive
# ============================================================
from google.colab import drive

SAVE_TO_DRIVE = False  # Set to True to save outputs

if SAVE_TO_DRIVE:
    drive.mount('/content/drive')
    
    import shutil
    drive_dir = "/content/drive/MyDrive/llm-finetune-outputs"
    os.makedirs(drive_dir, exist_ok=True)
    
    # Copy outputs
    for folder in ["./outputs", "./test_output", "./eval_results", "./data/processed"]:
        if os.path.exists(folder):
            dst = os.path.join(drive_dir, os.path.basename(folder))
            if os.path.exists(dst):
                shutil.rmtree(dst)
            shutil.copytree(folder, dst)
            print(f"✅ Saved {folder} → {dst}")
    
    print(f"\n📁 All outputs saved to: {drive_dir}")
    print("   These persist even after your Colab session ends.")
else:
    print("💡 Set SAVE_TO_DRIVE = True to save outputs to Google Drive.")
    print("   Colab files are deleted when the session ends!")

---
## 🛠️ Troubleshooting Common Colab Issues

| Problem | Solution |
|---------|----------|
| **No GPU** | Runtime → Change runtime type → T4 GPU |
| **CUDA out of memory** | Reduce `batch_size` to 1, `max_seq_length` to 1024, `lora_rank` to 32 |
| **Unsloth install fails** | `!pip install --upgrade pip` then retry. Or use standard transformers (slower but works) |
| **bitsandbytes not found** | `!pip install bitsandbytes>=0.42.0` — may need CUDA 11.8+ |
| **HuggingFace model 403** | Some models (Llama) need HF token: `!huggingface-cli login --token YOUR_TOKEN` |
| **Session crashes** | Colab has time limits (free: ~4h, Pro: ~12h). Use `SAVE_TO_DRIVE` frequently |
| **wandb prompts** | Set `os.environ['WANDB_MODE']='disabled'` (already done in Step 1) |
| **Google Drive mount fails** | Re-run the drive mount cell and authorize again |

### Quick VRAM Fix for OOM
```python
# Add these to TrainingArguments if you get OOM:
per_device_train_batch_size = 1        # Minimum batch
gradient_accumulation_steps = 16       # Compensate with more accumulation
max_seq_length = 1024                  # Shorter sequences = less VRAM
# LoRA rank = 16                        # Smaller adapter
```